# Tinker with ModernBERT-large

ModernBERT is a **bidirectional encoder** trained with masked language modeling. It is **not** a chat model: it will not write a paragraph answer. Use it like BERT:

1. **Fill-mask** — hide a word with `[MASK]`
2. **Score options** — which MCQ / T-F sentence the model likes more
3. **Embeddings** — similar questions and passages sit close in vector space

Model card: [`answerdotai/ModernBERT-large`](https://huggingface.co/answerdotai/ModernBERT-large) (28 layers, ~395M params). Needs `transformers >= 4.48`.

If this notebook runs out of memory, change `MODEL_ID` to `answerdotai/ModernBERT-base`.

In [1]:
import torch
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer, pipeline

from pathlib import Path

MODEL_ID = str(Path("../models/modernbert-large").resolve())
print("Loading", MODEL_ID)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else (
    torch.float16 if device == "cuda" else torch.float32
)
print(device, dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
mlm = AutoModelForMaskedLM.from_pretrained(MODEL_ID, dtype=dtype, local_files_only=True).to(device).eval()
encoder = AutoModel.from_pretrained(MODEL_ID, dtype=dtype, local_files_only=True).to(device).eval()
fill = pipeline("fill-mask", model=mlm, tokenizer=tokenizer, device=0 if device == "cuda" else -1)
print("params (M):", sum(p.numel() for p in mlm.parameters()) / 1e6)
print("mask token:", tokenizer.mask_token)
print("max positions:", mlm.config.max_position_embeddings)

C:\Users\bhaga\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading D:\work\BTP\models\modernbert-large
cuda torch.bfloat16


Loading weights: 100%|██████████| 170/170 [00:00<00:00, 302.07it/s]
[transformers] ModernBertModel LOAD REPORT from: D:\work\BTP\models\modernbert-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


params (M): 395.881664
mask token: [MASK]
max positions: 8192


## 1. Fill-mask

This is the native ModernBERT interface. Swap in your own HEA sentences.

In [2]:
from pprint import pprint

pprint(fill("The capital of France is [MASK].", top_k=5))
pprint(fill("Spark plasma [MASK] consolidates refractory high-entropy alloys.", top_k=5))
pprint(fill("Yield strength often increases as grain size [MASK].", top_k=5))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'score': 0.984375,
  'sequence': 'The capital of France is Paris.',
  'token': 7785,
  'token_str': ' Paris'},
 {'score': 0.00848388671875,
  'sequence': 'The capital of France is Nice.',
  'token': 29902,
  'token_str': ' Nice'},
 {'score': 0.005828857421875,
  'sequence': 'The capital of France is Lyon.',
  'token': 42268,
  'token_str': ' Lyon'},
 {'score': 0.00115203857421875,
  'sequence': 'The capital of France is France.',
  'token': 6181,
  'token_str': ' France'},
 {'score': 0.000423431396484375,
  'sequence': 'The capital of France is Tours.',
  'token': 49871,
  'token_str': ' Tours'}]
[{'score': 0.1484375,
  'sequence': 'Spark plasma treatment consolidates refractory high-entropy '
              'alloys.',
  'token': 1971,
  'token_str': ' treatment'},
 {'score': 0.05126953125,
  'sequence': 'Spark plasma technology consolidates refractory high-entropy '
              'alloys.',
  'token': 4302,
  'token_str': ' technology'},
 {'score': 0.048095703125,
  'sequence': 'Spar

## 2. "Ask a question" the BERT way

Turn the question into a cloze, or rank complete answer sentences.

In [3]:
pprint(fill(
    "The Hall-Petch relation says that smaller grains usually give [MASK] strength.",
    top_k=8,
))

[{'score': 0.4609375,
  'sequence': 'The Hall-Petch relation says that smaller grains usually give '
              'more strength.',
  'token': 625,
  'token_str': ' more'},
 {'score': 0.1923828125,
  'sequence': 'The Hall-Petch relation says that smaller grains usually give '
              'greater strength.',
  'token': 3687,
  'token_str': ' greater'},
 {'score': 0.1318359375,
  'sequence': 'The Hall-Petch relation says that smaller grains usually give '
              'less strength.',
  'token': 1679,
  'token_str': ' less'},
 {'score': 0.033447265625,
  'sequence': 'The Hall-Petch relation says that smaller grains usually give '
              'better strength.',
  'token': 1805,
  'token_str': ' better'},
 {'score': 0.0147705078125,
  'sequence': 'The Hall-Petch relation says that smaller grains usually give '
              'higher strength.',
  'token': 2169,
  'token_str': ' higher'},
 {'score': 0.0147705078125,
  'sequence': 'The Hall-Petch relation says that smaller grains usu

In [4]:
@torch.no_grad()
def mean_nll(text: str) -> float:
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    input_ids = enc["input_ids"].to(device)
    attn = enc["attention_mask"].to(device)
    special = set(tokenizer.all_special_ids)
    ids = input_ids[0]
    positions = [i for i, t in enumerate(ids.tolist()) if t not in special and attn[0, i] == 1]
    losses = []
    for i in positions:
        masked = input_ids.clone()
        masked[0, i] = tokenizer.mask_token_id
        logp = torch.log_softmax(mlm(input_ids=masked, attention_mask=attn).logits[0, i].float(), dim=-1)
        losses.append(-logp[ids[i]].item())
    return sum(losses) / len(losses)

stem = "According to Hall-Petch-type reasoning, finer grains typically"
options = [
    "increase yield strength and hardness.",
    "decrease yield strength and hardness.",
    "have no effect on mechanical strength.",
]
ranked = sorted(((mean_nll(f"{stem} {o}"), o) for o in options), key=lambda x: x[0])
for nll, o in ranked:
    print(f"{nll:.3f}  {o}")

2.334  have no effect on mechanical strength.
2.490  increase yield strength and hardness.
2.506  decrease yield strength and hardness.


## 3. Embeddings (what you will later feed to a prediction head)

In [5]:
@torch.no_grad()
def embed(text: str):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    enc = {k: v.to(device) for k, v in enc.items()}
    h = encoder(**enc).last_hidden_state
    m = enc["attention_mask"].unsqueeze(-1)
    pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
    return torch.nn.functional.normalize(pooled.float(), dim=-1)[0]

q = embed("How does grain size affect hardness after spark plasma sintering?")
passages = [
    "Finer equiaxed grains after SPS raise hardness via a Hall-Petch-type effect.",
    "The capital of France is Paris.",
]
for p in passages:
    print(f"{torch.dot(q, embed(p)).item():.4f}  {p}")

0.8660  Finer equiaxed grains after SPS raise hardness via a Hall-Petch-type effect.
0.8053  The capital of France is Paris.


Inspect the config: hidden size, layers, vocab. This is the "nook and cranny" view your professor asked for.

In [6]:
c = mlm.config
print({
    "model_type": c.model_type,
    "hidden_size": c.hidden_size,
    "num_hidden_layers": c.num_hidden_layers,
    "num_attention_heads": c.num_attention_heads,
    "intermediate_size": c.intermediate_size,
    "vocab_size": c.vocab_size,
    "max_position_embeddings": c.max_position_embeddings,
})

{'model_type': 'modernbert', 'hidden_size': 1024, 'num_hidden_layers': 28, 'num_attention_heads': 16, 'intermediate_size': 2624, 'vocab_size': 50368, 'max_position_embeddings': 8192}
